# Madhya Pradesh — Weekly Mandi Summary (State-Level, Bilingual)

Generates ONE state-level weekly mandi summary for Madhya Pradesh (not a per-district brief),
English first and Hindi second, from CSV/PDF exports of Agmarknet's Daily Price and Arrival
Report. Scope is exactly the confirmed problem statement:

1. Overall mandi arrival summary for the reporting period (total arrivals, week-on-week change).
2. Market reporting compliance — bucketed by number of reporting days (0 / 1-2 / 3-4 / 5-6 / 7),
   NOT a row per market (350+ markets can't be listed individually).
3. Top traded commodities by arrival quantity.
4. Week-on-week commodity price comparison.
5. Top 3 commodities by price increase, top 3 by price decrease.
6. Market reporting insight: which single market reported the most days, and how many markets
   reported 5-6 days.

Explicitly OUT of scope for this pass (per direct confirmation): a full commodity-by-commodity
min/max/modal price table, a "major commodities" (wheat/rice/maize/horticulture) special section,
and a separate not-submitted market list (covered by the 0-day compliance band instead).

**Accuracy rule carried over from the plan unchanged**: every number in the summary is computed
directly from the source rows in code, in this notebook. The narrative-writing step below is a
fixed, hand-authored template with numeric slots — not a free-generation model call — specifically
so the English and Hindi text can never state a number that doesn't match the fact sheet. If this
is later swapped for an actual open-source narration model (Qwen3, per the plan), the fact sheet
this notebook produces is exactly what that model would be grounded to, and a numeric diff-checker
would need to sit between the model and publication — see the closing note at the bottom.

**How to use**: drop CSV exports into `data/csv/` and PDF exports into `data/pdf/` — both the
target week's files and the prior week's files (needed for week-on-week comparison) — then run
all cells.

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timedelta

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 0. Paths & config

In [ ]:
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
CSV_DIR = DATA_DIR / "csv"
PDF_DIR = DATA_DIR / "pdf"
OUTPUT_DIR = BASE_DIR / "output"

for d in (CSV_DIR, PDF_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

STATE_NAME = "Madhya Pradesh"

# Set explicitly once real data is loaded, or leave None to auto-detect from the data
# (max arrival_date found = week_end, week_start = week_end - 6 days).
WEEK_END_OVERRIDE = None  # e.g. "2026-07-23"

# How many commodities to show in the "top traded by arrival" table (Image 2 style).
TOP_N_COMMODITIES = 10

print(f"CSV inputs expected in: {CSV_DIR}")
print(f"PDF inputs expected in: {PDF_DIR}")
print(f"Output written to:      {OUTPUT_DIR}")
print(f"CSV files found now:  {sorted(p.name for p in CSV_DIR.glob('*.csv'))}")
print(f"PDF files found now:  {sorted(p.name for p in PDF_DIR.glob('*.pdf'))}")

## 1. Field aliases

Canonical schema this pipeline works in, and every raw column-name variant seen so far that maps
to it. `arrival_qty` (tonnes/quintals, whatever unit the source states) is carried through
untouched — extend this dict, don't special-case a loader, the moment a new file uses a header not
listed here (the loader will print any *unmapped columns* it hits rather than silently dropping
them).

In [ ]:
FIELD_ALIASES = {
    "state": ["state", "State", "State/UT", "state_name", "State Name"],
    "district": ["district", "District", "district_name", "District Name"],
    "market": ["market", "Market", "market_name", "Market Name"],
    "commodity": ["commodity", "Commodity", "cmdt_name", "Commodity Name"],
    "commodity_group": ["commodity_group", "Commodity Group", "cmdt_grp_name", "Group"],
    "variety": ["variety", "Variety", "variety_name"],
    "grade": ["grade", "Grade", "grade_name"],
    "arrival_date": [
        "arrival_date", "Arrival_Date", "Reported Date", "reported_date",
        "Price Date", "Arrival Date", "Date",
    ],
    "min_price": ["min_price", "Min_Price", "Min Price", "Minimum Price (Rs./Quintal)"],
    "max_price": ["max_price", "Max_Price", "Max Price", "Maximum Price (Rs./Quintal)"],
    "modal_price": [
        "modal_price", "Modal_Price", "Modal Price", "model_price",
        "Modal Price (Rs./Quintal)",
    ],
    "price_unit": ["price_unit", "unit_name_price", "Price Unit"],
    "arrival_qty": [
        "arrival_qty", "Arrivals (Tonnes)", "Arrival Quantity", "arrival_quantity",
        "Arrivals", "Arrival Qty",
    ],
    "arrival_unit": ["arrival_unit", "unit_name_arrival", "Arrival Unit"],
}

CANONICAL_COLUMNS = list(FIELD_ALIASES.keys())

_ALIAS_LOOKUP = {
    alias.strip().lower(): canonical
    for canonical, aliases in FIELD_ALIASES.items()
    for alias in aliases
}


def map_columns(df: pd.DataFrame, source_label: str):
    """Rename df's columns to the canonical schema. Unmapped columns are kept
    under their original name (not dropped) and reported, so nothing the
    source provides is silently lost."""
    rename_map = {}
    unmapped = []
    for col in df.columns:
        key = str(col).strip().lower()
        if key in _ALIAS_LOOKUP:
            rename_map[col] = _ALIAS_LOOKUP[key]
        else:
            unmapped.append(col)
    renamed = df.rename(columns=rename_map)
    if unmapped:
        print(f"[{source_label}] unmapped columns (kept as-is): {unmapped}")
    return renamed, unmapped

## 2. CSV ingestion

In [ ]:
def load_csv_dir(csv_dir: Path) -> pd.DataFrame:
    frames = []
    for path in sorted(csv_dir.glob("*.csv")):
        try:
            raw = pd.read_csv(path, dtype=str)
        except Exception as exc:
            print(f"[CSV] FAILED to read {path.name}: {exc}")
            continue
        mapped, _ = map_columns(raw, path.name)
        mapped["_source_file"] = path.name
        frames.append(mapped)
        print(f"[CSV] loaded {path.name}: {len(mapped)} rows")
    if not frames:
        return pd.DataFrame(columns=CANONICAL_COLUMNS + ["_source_file"])
    return pd.concat(frames, ignore_index=True, sort=False)


csv_records = load_csv_dir(CSV_DIR)
print(f"\nTotal CSV rows loaded: {len(csv_records)}")
csv_records.head()

## 3. PDF ingestion

Rows that don't extract cleanly (merged/malformed cells) are collected in `pdf_parse_errors`
rather than silently skipped, so a parsing gap is visible instead of just missing from output.

In [ ]:
import pdfplumber


def load_pdf_dir(pdf_dir: Path):
    frames = []
    parse_errors = []
    for path in sorted(pdf_dir.glob("*.pdf")):
        try:
            with pdfplumber.open(path) as pdf:
                page_tables = []
                for page_num, page in enumerate(pdf.pages, start=1):
                    table = page.extract_table()
                    if not table or len(table) < 2:
                        parse_errors.append((path.name, page_num, "no table extracted"))
                        continue
                    header, *rows = table
                    try:
                        df = pd.DataFrame(rows, columns=header)
                    except Exception as exc:
                        parse_errors.append((path.name, page_num, f"header/row mismatch: {exc}"))
                        continue
                    page_tables.append(df)
                if not page_tables:
                    print(f"[PDF] {path.name}: no tables extracted on any page")
                    continue
                combined = pd.concat(page_tables, ignore_index=True, sort=False)
        except Exception as exc:
            parse_errors.append((path.name, None, f"file open/parse failed: {exc}"))
            print(f"[PDF] FAILED to read {path.name}: {exc}")
            continue
        mapped, _ = map_columns(combined, path.name)
        mapped["_source_file"] = path.name
        frames.append(mapped)
        print(f"[PDF] loaded {path.name}: {len(mapped)} rows")
    if not frames:
        empty = pd.DataFrame(columns=CANONICAL_COLUMNS + ["_source_file"])
    else:
        empty = pd.concat(frames, ignore_index=True, sort=False)
    return empty, parse_errors


pdf_records, pdf_parse_errors = load_pdf_dir(PDF_DIR)
print(f"\nTotal PDF rows loaded: {len(pdf_records)}")
if pdf_parse_errors:
    print(f"PDF parse errors ({len(pdf_parse_errors)}) — pages that did NOT make it into pdf_records:")
    for fname, page_num, reason in pdf_parse_errors:
        print(f"  - {fname} (page {page_num}): {reason}")
pdf_records.head()

## 4. Combine, clean, normalize types

- parses `arrival_date` (tries multiple formats — CSV vs PDF exports have used different ones)
- converts price/quantity fields to floats — a missing/unparseable value stays `None`/`NaN`, it is
  never coerced to 0
- blank district/market/commodity cells stay genuinely missing rather than becoming the literal
  string `"nan"` (a real bug caught during testing — `astype(str)` on a missing cell silently
  produces `"nan"`, which then gets counted as a fake category downstream)
- restricts to Madhya Pradesh rows (defensive, in case an export contains other states)

In [ ]:
DATE_FORMATS = ["%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d", "%d %b %Y", "%d-%b-%Y"]


def parse_date(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    value = str(value).strip()
    if not value or value.upper() in ("NA", "N/A"):
        return None
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    return None


def to_float(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).replace(",", "").strip()
    if text == "" or text.upper() in ("NA", "N/A"):
        return None
    try:
        return float(text)
    except ValueError:
        return None


def clean_str_col(series):
    return series.where(series.notna(), None).apply(lambda v: str(v).strip() if v is not None else None)


raw = pd.concat([csv_records, pdf_records], ignore_index=True, sort=False)
for col in CANONICAL_COLUMNS:
    if col not in raw.columns:
        raw[col] = None

date_parse_errors = []
raw["arrival_date_parsed"] = raw["arrival_date"].apply(parse_date)
bad_dates = raw[raw["arrival_date"].notna() & raw["arrival_date_parsed"].isna()]
for idx, row in bad_dates.iterrows():
    date_parse_errors.append((row.get("_source_file"), idx, row.get("arrival_date")))

for col in ("min_price", "max_price", "modal_price", "arrival_qty"):
    raw[col] = raw[col].apply(to_float)

raw["district"] = clean_str_col(raw["district"])
raw["market"] = clean_str_col(raw["market"])
raw["commodity"] = clean_str_col(raw["commodity"])

if "state" in raw.columns and raw["state"].notna().any():
    before = len(raw)
    raw = raw[raw["state"].astype(str).str.strip().str.lower() == STATE_NAME.lower()]
    print(f"Filtered to state={STATE_NAME!r}: {before} -> {len(raw)} rows")

print(f"Total combined rows: {len(raw)}")
if date_parse_errors:
    print(f"{len(date_parse_errors)} rows had an unparseable arrival_date — excluded from date-window aggregation:")
    for src, idx, val in date_parse_errors[:20]:
        print(f"  - source={src} row={idx}: {val!r}")

raw.head()

## 5. Reporting week window

`week_end` defaults to the latest parsed date in the data (override with `WEEK_END_OVERRIDE`
above). `week_start` is `week_end - 6 days`. The prior 7-day window is used for week-on-week
comparison in sections 8-9 — only populated if the data actually contains rows in that window.

In [ ]:
if WEEK_END_OVERRIDE:
    week_end = datetime.strptime(WEEK_END_OVERRIDE, "%Y-%m-%d").date()
else:
    valid_dates = raw["arrival_date_parsed"].dropna()
    week_end = valid_dates.max() if not valid_dates.empty else None

if week_end is None:
    print("No valid dates found yet — load data into data/csv or data/pdf and re-run from section 2.")
    week_start = prior_week_start = prior_week_end = None
    current_week_df = raw.iloc[0:0]
    prior_week_df = raw.iloc[0:0]
else:
    week_start = week_end - timedelta(days=6)
    prior_week_end = week_start - timedelta(days=1)
    prior_week_start = prior_week_end - timedelta(days=6)
    print(f"Current week: {week_start} to {week_end}")
    print(f"Prior week:   {prior_week_start} to {prior_week_end}")

    current_week_df = raw[raw["arrival_date_parsed"].between(week_start, week_end)]
    prior_week_df = raw[raw["arrival_date_parsed"].between(prior_week_start, prior_week_end)]
    print(f"Rows in current week: {len(current_week_df)}")
    print(f"Rows in prior week:   {len(prior_week_df)}")

## 6. Market reporting compliance (state-wide, bucketed — not a 350-row list)

The market roster is defined as *every market that appears at least once in the loaded data* —
the documented fallback for when an independently-sourced canonical 240-340+ market list isn't
available (same fallback as the underlying plan's risk register). Each market's `reporting_days`
in the current week is bucketed into the same bands shown in the reference dashboard: 0, 1-2, 3-4,
5-6, 7 days. `"0 days"` is exactly the not-submitted set — that's why there's no separate
not-submitted list elsewhere in this notebook.

In [ ]:
def reporting_band(days: int) -> str:
    if days <= 0:
        return "0 days"
    if days <= 2:
        return "1-2 days"
    if days <= 4:
        return "3-4 days"
    if days <= 6:
        return "5-6 days"
    return "7 days"


def compute_market_compliance(df: pd.DataFrame):
    roster = sorted(m for m in df["market"].dropna().unique())
    per_market_days = {}
    for market, g in df.groupby("market"):
        per_market_days[market] = g["arrival_date_parsed"].nunique()
    for market in roster:
        per_market_days.setdefault(market, 0)

    band_counts = defaultdict(int)
    for market in roster:
        band_counts[reporting_band(per_market_days[market])] += 1

    band_order = ["0 days", "1-2 days", "3-4 days", "5-6 days", "7 days"]
    bands = [{"band": b, "market_count": band_counts.get(b, 0)} for b in band_order]

    top_market = max(per_market_days.items(), key=lambda kv: (kv[1], kv[0]), default=(None, 0))

    return {
        "markets_in_roster": len(roster),
        "markets_reporting_at_least_once": sum(1 for d in per_market_days.values() if d > 0),
        "compliance_bands": bands,
        "markets_reporting_5_to_6_days": band_counts.get("5-6 days", 0),
        "markets_reporting_all_7_days": band_counts.get("7 days", 0),
        "markets_not_reporting": band_counts.get("0 days", 0),
        "top_reporting_market": top_market[0],
        "top_reporting_market_days": top_market[1],
        "roster_caveat": (
            "Roster = every market appearing at least once in the loaded CSV/PDF exports for "
            "this state, not an independently sourced canonical market list (~240-340+ markets "
            "for Madhya Pradesh per Agmarknet). If the loaded files don't cover a market for the "
            "whole window, it will show as 0 days here even if it reports normally outside this "
            "data pull."
        ),
    }


market_compliance = compute_market_compliance(current_week_df) if week_end else None
market_compliance

## 7. Top traded commodities by arrival quantity (state-wide)

Ranked by arrival volume when `arrival_qty` is present in the loaded data; falls back to
price-quote count with an explicit flag if arrival quantity isn't available in the source files —
never silently substituted for tonnage. Modal price shown is the simple mean of modal prices
across all rows for that commodity in the window (not a full min/max/modal breakdown — that level
of detail is out of scope per the confirmed requirements).

In [ ]:
def compute_top_commodities(current_df: pd.DataFrame, prior_df: pd.DataFrame, top_n: int):
    has_tonnage = current_df["arrival_qty"].notna().any()

    if has_tonnage:
        by_commodity = (
            current_df.dropna(subset=["arrival_qty"])
            .groupby("commodity")["arrival_qty"].sum()
            .sort_values(ascending=False)
        )
        total = by_commodity.sum()
        ranking_basis = "arrival_qty"
    else:
        by_commodity = current_df["commodity"].value_counts()
        total = by_commodity.sum()
        ranking_basis = "price_quote_count (arrival_qty not present in loaded data)"

    prior_by_commodity = (
        prior_df.dropna(subset=["arrival_qty"]).groupby("commodity")["arrival_qty"].sum()
        if has_tonnage and not prior_df.empty else pd.Series(dtype=float)
    )

    rows = []
    for commodity, value in by_commodity.head(top_n).items():
        if commodity is None:
            continue
        modal_prices = current_df.loc[current_df["commodity"] == commodity, "modal_price"].dropna()
        markets_trading = current_df.loc[current_df["commodity"] == commodity, "market"].nunique()

        wow_arrival_pct = None
        if has_tonnage and commodity in prior_by_commodity.index and prior_by_commodity[commodity]:
            wow_arrival_pct = round((value - prior_by_commodity[commodity]) / prior_by_commodity[commodity] * 100, 1)

        rows.append({
            "commodity": commodity,
            "arrival_value": round(float(value), 2),
            "arrival_ranking_basis": ranking_basis,
            "share_pct_of_state_arrivals": round(float(value) / total * 100, 1) if total else None,
            "markets_trading": int(markets_trading),
            "modal_price_mean": round(float(modal_prices.mean()), 2) if len(modal_prices) else None,
            "wow_arrival_pct_change": wow_arrival_pct,
        })

    return {
        "ranking_basis": ranking_basis,
        "top_commodities": rows,
        "total_commodities_traded": int(current_df["commodity"].dropna().nunique()),
    }


top_commodities = compute_top_commodities(current_week_df, prior_week_df, TOP_N_COMMODITIES) if week_end else None
top_commodities

## 8. Overall arrival summary (state totals + week-on-week change)

In [ ]:
def compute_overall_arrivals(current_df: pd.DataFrame, prior_df: pd.DataFrame):
    has_tonnage = current_df["arrival_qty"].notna().any()
    if has_tonnage:
        current_total = float(current_df["arrival_qty"].dropna().sum())
        prior_total = float(prior_df["arrival_qty"].dropna().sum()) if not prior_df.empty else None
        basis = "arrival_qty"
    else:
        current_total = int(len(current_df))
        prior_total = int(len(prior_df)) if not prior_df.empty else None
        basis = "price_quote_count (arrival_qty not present in loaded data)"

    wow_pct = None
    if prior_total:
        wow_pct = round((current_total - prior_total) / prior_total * 100, 1)

    return {
        "total_arrivals": round(current_total, 2) if has_tonnage else current_total,
        "total_arrivals_basis": basis,
        "prior_week_total_arrivals": round(prior_total, 2) if (prior_total is not None and has_tonnage) else prior_total,
        "wow_pct_change": wow_pct,
    }


overall_arrivals = compute_overall_arrivals(current_week_df, prior_week_df) if week_end else None
overall_arrivals

## 9. Week-on-week price change — top 3 gainers, top 3 decliners (state-wide)

Compares each commodity's mean modal price between the current and prior week. Only commodities
present in **both** weeks are ranked — no interpolation for a commodity missing from either week.

In [ ]:
def compute_price_change(current_df: pd.DataFrame, prior_df: pd.DataFrame):
    cur = current_df.dropna(subset=["modal_price"]).groupby("commodity")["modal_price"].mean()
    prior = prior_df.dropna(subset=["modal_price"]).groupby("commodity")["modal_price"].mean()
    common = cur.index.intersection(prior.index)
    if len(common) == 0:
        return {"available": False, "reason": "no overlapping commodities between current and prior week", "top_gainers": [], "top_decliners": []}

    pct_change = ((cur[common] - prior[common]) / prior[common] * 100).round(2)
    gainers_only = pct_change[pct_change > 0].sort_values(ascending=False)
    decliners_only = pct_change[pct_change < 0].sort_values()

    def _rows(series):
        return [
            {"commodity": k, "pct_change": v, "current_modal_price": round(cur[k], 2), "prior_modal_price": round(prior[k], 2)}
            for k, v in series.head(3).items()
        ]

    top_gainers = _rows(gainers_only)
    top_decliners = _rows(decliners_only)
    return {"available": True, "top_gainers": top_gainers, "top_decliners": top_decliners}


if week_end and not prior_week_df.empty:
    price_change = compute_price_change(current_week_df, prior_week_df)
elif week_end:
    print("No prior-week data loaded — price_change left unavailable (need a second week of "
          "CSV/PDF exports in data/csv or data/pdf).")
    price_change = {"available": False, "reason": "prior week not loaded", "top_gainers": [], "top_decliners": []}
else:
    price_change = None

price_change

## 10. Assemble the state fact sheet

The only object the narrative step below reads from. Nothing past this point computes a new
number — sections 11-12 only arrange these numbers into sentences.

In [ ]:
state_fact_sheet = {
    "state": STATE_NAME,
    "week_start": str(week_start) if week_start else None,
    "week_end": str(week_end) if week_end else None,
    "overall_arrivals": overall_arrivals,
    "market_compliance": market_compliance,
    "top_commodities": top_commodities,
    "price_change": price_change,
}

out_path = OUTPUT_DIR / "madhya_pradesh_state_fact_sheet.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(state_fact_sheet, f, indent=2, ensure_ascii=False, default=str)

print(f"Wrote fact sheet to {out_path}")
json.loads(json.dumps(state_fact_sheet, default=str))

## 11. English narrative

A fixed template with numeric slots — every figure it prints comes straight from
`state_fact_sheet`, so it cannot state a number that disagrees with section 10. This is the
stand-in for the plan's "AI model converts verified data into narrative" step; it produces the
narration deterministically rather than via a live model call, since no open-source narration
model is standing up in this notebook environment. Swapping in Qwen3 later means: prompt it with
`state_fact_sheet` as the *only* input, and gate its output against these exact numbers before
publishing.

In [ ]:
def fmt_pct(value):
    if value is None:
        return "not available (no prior-week data loaded)"
    sign = "+" if value >= 0 else ""
    return f"{sign}{value}%"


def narrate_english(fs: dict) -> str:
    if fs["week_start"] is None:
        return "No data loaded yet — nothing to narrate."

    oa = fs["overall_arrivals"]
    mc = fs["market_compliance"]
    tc = fs["top_commodities"]
    pc = fs["price_change"]

    unit_note = "" if oa["total_arrivals_basis"] == "arrival_qty" else " (price-quote count, arrival quantity not present in source files)"
    lines = []
    lines.append(
        f"Madhya Pradesh Weekly Mandi Summary — {fs['week_start']} to {fs['week_end']}"
    )
    lines.append("")
    lines.append(
        f"Total arrivals across Madhya Pradesh markets for the week were "
        f"{oa['total_arrivals']:,}{unit_note}, "
        f"a change of {fmt_pct(oa['wow_pct_change'])} over the previous week."
    )

    if tc["top_commodities"]:
        top = tc["top_commodities"][0]
        share = f", {top['share_pct_of_state_arrivals']}% of state arrivals" if top["share_pct_of_state_arrivals"] is not None else ""
        lines.append(
            f"{top['commodity']} was the most-traded commodity by arrival volume "
            f"({top['arrival_value']:,}{share}), traded across {top['markets_trading']} markets."
        )
        others = ", ".join(
            f"{r['commodity']} ({r['arrival_value']:,})" for r in tc["top_commodities"][1:5]
        )
        if others:
            lines.append(f"Other leading commodities by arrival volume: {others}.")

    if pc and pc.get("available"):
        gainers = "; ".join(
            f"{g['commodity']} {fmt_pct(g['pct_change'])} (Rs {g['current_modal_price']:,} from Rs {g['prior_modal_price']:,})"
            for g in pc["top_gainers"]
        )
        decliners = "; ".join(
            f"{d['commodity']} {fmt_pct(d['pct_change'])} (Rs {d['current_modal_price']:,} from Rs {d['prior_modal_price']:,})"
            for d in pc["top_decliners"]
        )
        lines.append(f"Largest week-on-week price increases: {gainers if gainers else 'none — no commodity gained price this week'}.")
        lines.append(f"Largest week-on-week price decreases: {decliners if decliners else 'none — no commodity lost price this week'}.")
    else:
        reason = (pc or {}).get("reason", "prior week not loaded")
        lines.append(f"Week-on-week price comparison is not available this week ({reason}).")

    lines.append(
        f"Of {mc['markets_in_roster']} market yards seen in the source data, "
        f"{mc['markets_reporting_at_least_once']} reported at least once this week. "
        f"{mc['markets_reporting_all_7_days']} markets reported on all 7 days, "
        f"{mc['markets_reporting_5_to_6_days']} reported on 5-6 days, and "
        f"{mc['markets_not_reporting']} filed no return for the week."
    )
    if mc["top_reporting_market"]:
        lines.append(
            f"{mc['top_reporting_market']} reported the most days ({mc['top_reporting_market_days']} of the week)."
        )
    lines.append("")
    lines.append(f"Note: {mc['roster_caveat']}")

    return "\n".join(lines)


english_brief = narrate_english(state_fact_sheet)
print(english_brief)

## 12. Hindi narrative

Written as a **parallel fixed template**, not a machine translation of the English text — same
numeric slots, same source numbers, authored with the plan's locked agri-terminology glossary
(गेहूँ=Wheat, मंडी=Mandi, आवक=Arrivals, मोडल मूल्य=Modal Price, क्विंटल=Quintal, टन=Tonnes). This
sidesteps needing IndicTrans2 (no GPU/model runtime in this notebook) while still guaranteeing the
Hindi figures can never drift from the English ones, since both templates read the same
`state_fact_sheet`. If IndicTrans2 is stood up later, it should translate the *English template
output*, not generate independently — same principle as the plan's translation step.

In [ ]:
GLOSSARY_NOTE = "गेहूँ=Wheat, मंडी=Mandi, आवक=Arrivals, मोडल मूल्य=Modal Price, क्विंटल=Quintal, टन=Tonnes"

# Fixed commodity-name glossary — extend as new commodities show up in the data.
# Falls back to the source (English) commodity name when not listed, rather than guessing a
# translation.
COMMODITY_NAME_HI = {
    "wheat": "गेहूँ", "soyabean": "सोयाबीन", "soybean": "सोयाबीन", "onion": "प्याज",
    "garlic": "लहसुन", "tomato": "टमाटर", "gram": "चना", "maize": "मक्का",
    "mustard": "सरसों", "potato": "आलू", "tur": "तुअर", "paddy": "धान",
    "lentil": "मसूर", "coriander": "धनिया", "rice": "चावल",
}


def commodity_hi(name):
    if not name:
        return name
    return COMMODITY_NAME_HI.get(str(name).strip().lower(), name)


def fmt_pct_hi(value):
    if value is None:
        return "उपलब्ध नहीं (पिछले सप्ताह का डेटा लोड नहीं किया गया)"
    sign = "+" if value >= 0 else ""
    return f"{sign}{value}%"


def narrate_hindi(fs: dict) -> str:
    if fs["week_start"] is None:
        return "अभी तक कोई डेटा लोड नहीं हुआ है।"

    oa = fs["overall_arrivals"]
    mc = fs["market_compliance"]
    tc = fs["top_commodities"]
    pc = fs["price_change"]

    unit_note = "" if oa["total_arrivals_basis"] == "arrival_qty" else " (मूल्य-उद्धरण गणना, स्रोत फ़ाइलों में आवक मात्रा उपलब्ध नहीं)"
    lines = []
    lines.append(f"मध्य प्रदेश साप्ताहिक मंडी सारांश — {fs['week_start']} से {fs['week_end']}")
    lines.append("")
    lines.append(
        f"मध्य प्रदेश की मंडियों में इस सप्ताह कुल आवक {oa['total_arrivals']:,}{unit_note} रही, "
        f"जो पिछले सप्ताह की तुलना में {fmt_pct_hi(oa['wow_pct_change'])} है।"
    )

    if tc["top_commodities"]:
        top = tc["top_commodities"][0]
        share = f", राज्य की आवक का {top['share_pct_of_state_arrivals']}%" if top["share_pct_of_state_arrivals"] is not None else ""
        lines.append(
            f"आवक मात्रा के आधार पर {commodity_hi(top['commodity'])} सबसे अधिक कारोबार वाली वस्तु रही "
            f"({top['arrival_value']:,}{share}), जिसका कारोबार {top['markets_trading']} मंडियों में हुआ।"
        )
        others = ", ".join(
            f"{commodity_hi(r['commodity'])} ({r['arrival_value']:,})" for r in tc["top_commodities"][1:5]
        )
        if others:
            lines.append(f"आवक मात्रा के आधार पर अन्य प्रमुख वस्तुएँ: {others}.")

    if pc and pc.get("available"):
        gainers = "; ".join(
            f"{commodity_hi(g['commodity'])} {fmt_pct_hi(g['pct_change'])} (रु {g['current_modal_price']:,}, पूर्व रु {g['prior_modal_price']:,})"
            for g in pc["top_gainers"]
        )
        decliners = "; ".join(
            f"{commodity_hi(d['commodity'])} {fmt_pct_hi(d['pct_change'])} (रु {d['current_modal_price']:,}, पूर्व रु {d['prior_modal_price']:,})"
            for d in pc["top_decliners"]
        )
        lines.append(f"साप्ताहिक आधार पर सबसे अधिक मूल्य वृद्धि: {gainers if gainers else 'कोई नहीं — इस सप्ताह किसी भी वस्तु की कीमत में वृद्धि नहीं हुई'}.")
        lines.append(f"साप्ताहिक आधार पर सबसे अधिक मूल्य गिरावट: {decliners if decliners else 'कोई नहीं — इस सप्ताह किसी भी वस्तु की कीमत में गिरावट नहीं हुई'}.")
    else:
        lines.append("इस सप्ताह मूल्य तुलना उपलब्ध नहीं है (पिछले सप्ताह का डेटा लोड नहीं किया गया)।")

    lines.append(
        f"स्रोत डेटा में दिखी {mc['markets_in_roster']} मंडियों में से, "
        f"{mc['markets_reporting_at_least_once']} मंडियों ने इस सप्ताह कम से कम एक बार रिपोर्ट की। "
        f"{mc['markets_reporting_all_7_days']} मंडियों ने सातों दिन रिपोर्ट की, "
        f"{mc['markets_reporting_5_to_6_days']} मंडियों ने 5-6 दिन रिपोर्ट की, और "
        f"{mc['markets_not_reporting']} मंडियों ने इस सप्ताह कोई रिपोर्ट दर्ज नहीं की।"
    )
    if mc["top_reporting_market"]:
        lines.append(
            f"{mc['top_reporting_market']} मंडी ने सबसे अधिक दिन रिपोर्ट की (सप्ताह के {mc['top_reporting_market_days']} दिन)।"
        )
    lines.append("")
    lines.append(f"शब्दावली: {GLOSSARY_NOTE}")

    return "\n".join(lines)


hindi_brief = narrate_hindi(state_fact_sheet)
print(hindi_brief)

## 13. Write the brief to disk

In [ ]:
en_path = OUTPUT_DIR / "madhya_pradesh_weekly_brief_en.txt"
hi_path = OUTPUT_DIR / "madhya_pradesh_weekly_brief_hi.txt"

with open(en_path, "w", encoding="utf-8") as f:
    f.write(english_brief)
with open(hi_path, "w", encoding="utf-8") as f:
    f.write(hindi_brief)

print(f"Wrote {en_path}")
print(f"Wrote {hi_path}")

## Not yet in this notebook (by design)

- **A real narration model (Qwen3)** in place of the fixed templates in sections 11-12, if the
  brief needs to read less formulaically. If swapped in, a **numeric diff-checker** must sit
  between the model and publication — extract every number the model writes and hard-match it
  against `state_fact_sheet`, exactly as the plan specifies. The templates above don't need this
  gate because they can't state a number they weren't given.
- **IndicTrans2** for Hindi, if machine translation of the English prose is wanted instead of the
  parallel hand-authored template — should translate the English *output*, not generate
  independently.
- **HHEM-2.1-Open grounding score** and **human review** — later verification steps, not part of
  computing the fact sheet.
- A canonical, independently-sourced Madhya Pradesh market list (to replace the
  data-derived roster caveat in section 6) if the market-compliance denominator needs to be exact
  rather than "every market seen in the loaded files."</br>